# Esperimento di Topic Labeling

In questo notebook confrontiamo diversi approcci per l'etichettatura automatica dei cluster individuati tramite HDBSCAN.

## Obiettivi:
1. Caricare i cluster e i documenti associati.
2. Applicare **YAKE**, **TextRank**, **c-TF-IDF** e **KeyBERT (Simulated)**.
3. Estrarre i documenti rappresentativi per ogni cluster.
4. Salvare i risultati comparativi nei metadata.

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import json

# Aggiungiamo src al path per importare le utility
sys.path.append(os.path.abspath("../../"))

from src.utils.topic_labeling import (
    extract_keywords_yake, 
    extract_keywords_textrank, 
    calculate_ctfidf, 
    extract_keywords_keybert,
    get_cluster_representative_docs
)

## 1. Caricamento Dati

In [2]:
df_processed = pd.read_parquet("../../data/processed/jmail_emails_processed.parquet")
df_clusters = pd.read_parquet("../../data/processed/email_cluster_assignments.parquet")
embeddings = np.load("../../data/embeddings/email_embeddings_bge-small-en-v1-5.npy")

df = df_processed.merge(df_clusters, on="id", how="inner")
print(f"Totale righe: {len(df)}")

Totale righe: 42471


## 2. Applicazione Multi-Algoritmo
Eseguiamo tutti i metodi per ogni cluster.

In [3]:
docs_per_cluster = df.groupby("cluster_id")["combined_text"].apply(lambda x: " ".join(x)).to_dict()
cluster_list = sorted([c for c in docs_per_cluster.keys()])

# Pre-calcolo c-TF-IDF
ctfidf_labels_dict = calculate_ctfidf(docs_per_cluster, top_n=10)

final_labels = {}

for cid in cluster_list:
    cluster_df = df[df["cluster_id"] == cid]
    # Campione per algoritmi lenti o basati su singolo testo
    sample_text = " ".join(cluster_df["combined_text"].head(50))
    
    # Simulazione Word Embeddings per KeyBERT (usiamo il centroide come proxy)
    cluster_emb = embeddings[cluster_df["embedding_row"]].mean(axis=0)
    # Mock per word embeddings: estraiamo parole dal sample e facciamo finta di avere i loro embedding
    words_in_sample = list(set(sample_text.split()))[:100]
    mock_word_embs = {w: np.random.randn(384) for w in words_in_sample}
    
    final_labels[str(cid)] = {
        "ctfidf": ctfidf_labels_dict.get(cid, []),
        "yake": extract_keywords_yake(sample_text, top_n=10),
        "textrank": extract_keywords_textrank(sample_text, top_n=10),
        "keybert_sim": extract_keywords_keybert(cluster_df["combined_text"].head(10).tolist(), cluster_emb, mock_word_embs, top_n=10)
    }

print("Labeling completato.")

/home/suga/Workspace/Projects/pizza-cluster/.venv/lib/python3.13/site-packages


Labeling completato.


## 3. Salvataggio Metadata

In [4]:
output_path = "../../data/metadata/cluster_labeling_metadata.json"
metadata = {
    "generated_at": pd.Timestamp.now().isoformat(),
    "algorithms": ["c-TF-IDF", "YAKE", "TextRank", "KeyBERT-Sim"],
    "n_clusters": len(final_labels),
    "cluster_labels": final_labels
}

with open(output_path, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Metadata aggiornati in {output_path}")

Metadata aggiornati in ../../data/metadata/cluster_labeling_metadata.json


## 4. Visualizzazione Comparativa

In [5]:
comparison_df = []
for cid in ["0", "3", "11"]:
    if cid in final_labels:
        row = {"cluster": cid}
        row.update(final_labels[cid])
        comparison_df.append(row)

display(pd.DataFrame(comparison_df))

,cluster,ctfidf,yake,textrank,keybert_sim
0,0,"[times, newyorktimesinfo, com, new, nytimes, y...","[Times, YORK, offer, Digital, time, Subscripti...","[New York Times, New York Times subscriptions,...","[relevant, access., LIMITED-TIME, big, holiday..."
1,3,"[alert, compliance, pcr, jeffrey, epstein, kyc...","[ALERT, Jeffrey, Epstein, KYC, PCR, CONFIDENTI...","[RDC ALERT, rdc alert, JJ JJ Litchford Associa...","[██████, prior, all, Passion, Epstein, ahead, ..."
2,11,"[pay, fraud, check, signature, pdf, ems, proce...","[Check, CONFIDENTIAL, Signature, EMS, Pay, PUR...","[Check Referrals, Check referrals, EFTA0135853...","[forbidden., earliest., 42959324-315[1], -M, 4..."
